In [1]:
from transformers import pipeline
import torch
import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer

In [4]:
import torch
import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer

# Constants for image normalization
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    """Build image transformation pipeline"""
    MEAN, STD = IMAGENET_MEAN, IMAGENET_STD
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD)
    ])
    return transform

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    """Find the closest aspect ratio from target ratios"""
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    """Dynamically preprocess image into tiles"""
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height
    
    # Calculate target ratios
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1) 
        for i in range(1, n + 1) 
        for j in range(1, n + 1) 
        if i * j <= max_num and i * j >= min_num
    )
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])
    
    # Find closest aspect ratio
    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size
    )
    
    # Calculate target dimensions
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]
    
    # Resize and split image
    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size
        )
        split_img = resized_img.crop(box)
        processed_images.append(split_img)
    
    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    
    return processed_images

def load_image(image_file, input_size=448, max_num=12):
    """Load and preprocess an image"""
    image = Image.open(image_file).convert('RGB')
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(image) for image in images]
    pixel_values = torch.stack(pixel_values)
    return pixel_values

# Load model and tokenizer
print("Loading model...")
path = "OpenGVLab/InternVL3-14B"
model = AutoModel.from_pretrained(
    path,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    use_flash_attn=True,
    trust_remote_code=True
).eval().cuda()

tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True, use_fast=False)

Loading model...


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [7]:
# Load and process image
print("Loading image...")
pixel_values = load_image('/export/home/ru63zus/repos/contrastive-skip-layer-guidance/experiments/flux_results_20250923_112436_text/A goblin librarian i/seed_1/slg_3.0_cfg_4.5.png', max_num=12).to(torch.bfloat16).cuda()

# Configure generation
generation_config = dict(max_new_tokens=1024, do_sample=True)

print("\n=== Single Image Query ===")
question = '''<image>\nYour a given an image that was generated using a text prompt. 
            The image contains multiple parts of text that should be visible in the image. 
            Please rate on a scale from 0 to 10 how well the image contains the visible text that is contained in the prompt. 
            A high score should be given to images that contain all text instances, contain text that is clearly visible, do not contain typos, and do not contain additional text. 
            A low score should be given to images where text is missing entirely, where letters are unreadable or made up, and where there are lots of typos. 
            The following text instances should all be clearly visible: \'SPELL INDEX\', \'Vol. 3: Memory Leaks\', \'Index of Forbidden Footnotes\', \'Chief Archivist Zog\'."
            Assign a score for each individual instance, and then average over all instances.'''
response = model.chat(tokenizer, pixel_values, question, generation_config)
print(f'User: {question}')
print(f'Assistant: {response}')

Loading image...

=== Single Image Query ===


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


User: <image>
Your a given an image that was generated using a text prompt. 
            The image contains multiple parts of text that should be visible in the image. 
            Please rate on a scale from 0 to 10 how well the image contains the visible text that is contained in the prompt. 
            A high score should be given to images that contain all text instances, contain text that is clearly visible, do not contain typos, and do not contain additional text. 
            A low score should be given to images where text is missing entirely, where letters are unreadable or made up, and where there are lots of typos. 
            The following text instances should all be clearly visible: 'SPELL INDEX', 'Vol. 3: Memory Leaks', 'Index of Forbidden Footnotes', 'Chief Archivist Zog'."
            Assign a score for each individual instance, and then average over all instances.
Assistant: 1. **SPELL INDEX**: 
   - Visible and clearly readable.
   - Score: 10

2. **Vol. 3: Memory 

In [16]:
# Load and process image
print("Loading image...")
pixel_values = load_image('/export/home/ru63zus/repos/contrastive-skip-layer-guidance/experiments/flux_results_20250924_074213_hands/Portrait of a young /seed_2/slg_2.0_cfg_1.0.png', max_num=12).to(torch.bfloat16).cuda()

# Configure generation
generation_config = dict(max_new_tokens=1024, do_sample=True)

print("\n=== Single Image Query ===")
question = '''<image>
You are a critical evaluator of AI-generated images. Analyze the hands in this image with extreme scrutiny.

First, carefully examine and describe:
1. How many hands are visible?
2. How many fingers does each hand have?
3. Are the proportions of fingers, palms, and wrists anatomically correct?
4. Do the joints (knuckles, wrist) bend naturally?
5. Are there any extra or missing fingers?
6. Are there any fingers that merge together or split unnaturally?
7. Do the hands have the correct anatomy (thumb on the right side, proper finger order)?
8. Are the fingernails present and correctly positioned?
9. Do the hands match the context (correct size relative to body, proper positioning)?
10. Are there any other anatomical impossibilities or artifacts?

Be HARSH in your evaluation. Real human hands have:
- Exactly 5 fingers per hand
- Proper joint articulation
- Consistent anatomy
- Natural proportions

Common AI failures include:
- Wrong number of fingers (6+ fingers or merged fingers)
- Fingers bending in impossible directions
- Missing or extra joints
- Hands that morph into other objects
- Inconsistent hand sizes
- Fingers that are too long/short/thick/thin

After your detailed analysis, assign a score from 0-100 where:
- 90-100: Perfect human hands, indistinguishable from a photograph
- 70-89: Minor issues (slight proportion problems, subtle artifacts)
- 50-69: Noticeable problems (weird finger positions, questionable anatomy)
- 30-49: Clear AI artifacts (wrong finger count, merged digits, impossible poses)
- 0-29: Severe problems (completely malformed hands, obvious failures)

Provide your analysis first, then give your final score.'''
response = model.chat(tokenizer, pixel_values, question, generation_config)
print(f'User: {question}')
print(f'Assistant: {response}')

Loading image...

=== Single Image Query ===


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


User: <image>
You are a critical evaluator of AI-generated images. Analyze the hands in this image with extreme scrutiny.

First, carefully examine and describe:
1. How many hands are visible?
2. How many fingers does each hand have?
3. Are the proportions of fingers, palms, and wrists anatomically correct?
4. Do the joints (knuckles, wrist) bend naturally?
5. Are there any extra or missing fingers?
6. Are there any fingers that merge together or split unnaturally?
7. Do the hands have the correct anatomy (thumb on the right side, proper finger order)?
8. Are the fingernails present and correctly positioned?
9. Do the hands match the context (correct size relative to body, proper positioning)?
10. Are there any other anatomical impossibilities or artifacts?

Be HARSH in your evaluation. Real human hands have:
- Exactly 5 fingers per hand
- Proper joint articulation
- Consistent anatomy
- Natural proportions

Common AI failures include:
- Wrong number of fingers (6+ fingers or merged fi

In [23]:
# Load and process image
print("Loading image...")
pixel_values = load_image('/export/home/ru63zus/repos/contrastive-skip-layer-guidance/experiments/flux_layer_ablation_20250927_075249/a moody city street /seed_1/layer_0.png', max_num=12).to(torch.bfloat16).cuda()

# Configure generation
generation_config = dict(max_new_tokens=1024, do_sample=True)

print("\n=== Single Image Query ===")
question = '''\n<image> \nPlease assess how aesthetically pleasing this image is. After providing your arguments, end your answer with a single score from 0 to 10.'''
response = model.chat(tokenizer, pixel_values, question, generation_config)
print(f'User: {question}')
print(f'Assistant: {response}')

Loading image...

=== Single Image Query ===


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


User: 
<image> 
Please assess how aesthetically pleasing this image is. After providing your arguments, end your answer with a single score from 0 to 10.
Assistant: The image depicts a street scene at sunset, with the sun casting a spotlight effect between the buildings, creating a dramatic and visually appealing atmosphere. Several elements contribute to its aesthetic value:

1. **Lighting**: The sun’s position creates a striking contrast, enhancing the moodiness of the cityscape.
2. **Symmetry**: The buildings flank the street symmetrically, leading the viewer’s eye toward the sun.
3. **Color Palette**: The warm tones of the sunset are complemented by the cool tones of the sky, adding depth and richness to the image.
4. **Composition**: The perspective draws the viewer into the scene, effectively using the rule of thirds.
5. **Atmospheric Effects**: The presence of clouds and the play of light add dynamism and interest.

While the image benefits from these elements, the pixelated qua